In [5]:
"""
Range-Inventory-Ladder backtester (NonKYC XMR/USDT) -- v3
=========================================================
New in v3:
  * SELECTABLE INTERVAL (default 5m). Finer bars pin limit fills far more
    precisely than hourly and shrink the intrabar-path assumption's impact.
  * COOLDOWN FIX: cooldown_time is now converted to *bars* from the bar size.
    On 5m data, your 3600s cooldown = 12 bars (was wrongly 1 bar/5min in v2,
    which would let rungs re-arm 12x too fast).

Carries forward from v2:
  * CASH CONSTRAINT on buys (no negative cash / >100% inventory).
  * Whatever MEXC returns is conformed to the requested interval grid.

Same ladder model: explicit buy/sell bands, dead zone, shared inventory pool
seeded with claimed_base_value_quote. amounts_pct interpretation still a switch
-- verify the printed $size column against your live orders.
"""

import requests, time

# ============================ CONFIG SWITCHES ============================
DAYS         = 360
COIN         = "XMRUSDT"
INTERVAL     = "5m"           # "1m" "5m" "15m" "30m" "60m" "4h" "1d"
FEE_RATE     = 0.002          # NonKYC maker fee (0.2%)
AMOUNTS_MODE = "literal_pct"  # "literal_pct" | "weights"
REARM        = True           # False = each rung fills at most once
PATH_CONV    = "standard"     # "standard" | "pessimistic"
# ========================================================================

_SECONDS = {"1m": 60, "5m": 300, "15m": 900, "30m": 1800,
            "60m": 3600, "4h": 14400, "1d": 86400}
INTERVAL_SECONDS = _SECONDS[INTERVAL]


CFG_V1 = {
    "name": "V1 (current)",
    "total_amount_quote": 600,
    "claimed_base_value_quote": 300,
    "cooldown_time": 3600,
    "buy_prices":   [328, 324, 321, 318, 315, 312, 305],
    "buy_amounts_pct":  [0.5, 1, 1, 4, 3.5, 1, 0.5],
    "sell_prices":  [331.5, 335, 340, 345, 350, 355, 360, 370, 380],
    "sell_amounts_pct": [0.5, 1, 1, 2, 2, 1, 1, 1, 1],
}



CFG_V2 = {
    "name": "V2 (backtest-tuned)",
    "total_amount_quote": 600,
    "claimed_base_value_quote": 300,
    "cooldown_time": 3600,
    "buy_prices":   [310.267, 308.39, 306.521, 304.649, 302.776],
    "buy_amounts_pct":  [12.3, 15.3, 19.1, 23.7, 29.5],
    "sell_prices":  [325.874, 339.608, 353.342, 367.077, 380.811],
    "sell_amounts_pct": [12.3, 15.3, 19.1, 23.7, 29.5],
}


def parse_config(text, name="custom"):
    cfg = {"name": name}
    flt = lambda s: [float(x) for x in s.split(",") if x.strip() != ""]
    for line in text.splitlines():
        line = line.split("#")[0].strip()
        if ":" not in line:
            continue
        k, v = [p.strip() for p in line.split(":", 1)]
        if k in ("total_amount_quote", "claimed_base_value_quote", "cooldown_time"):
            try: cfg[k] = float(v)
            except ValueError: pass
        elif k in ("buy_prices", "sell_prices", "buy_amounts_pct", "sell_amounts_pct"):
            cfg[k] = flt(v)
    cfg.setdefault("claimed_base_value_quote", 0.0)
    cfg.setdefault("cooldown_time", 0.0)
    return cfg


def fetch_mexc(symbol, days, interval):
    end   = int(time.time() * 1000)
    start = end - days * 24 * 3600 * 1000
    out, cur = {}, start
    for _ in range(200):
        r = requests.get("https://api.mexc.com/api/v3/klines",
                         params={"symbol": symbol, "interval": interval,
                                 "startTime": cur, "endTime": end, "limit": 500}, timeout=20)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        for k in batch:
            out[k[0]] = k
        latest = max(k[0] for k in batch)
        if latest <= cur or latest >= end or len(batch) < 500:
            break
        cur = latest + 1
        time.sleep(0.2)
    rows = [out[t] for t in sorted(out)]
    return [(int(k[0]), float(k[1]), float(k[2]), float(k[3]), float(k[4])) for k in rows]


def resample(rows, seconds):
    """Conform arbitrary bars to a `seconds` grid -> [(o,h,l,c), ...]."""
    ms = seconds * 1000
    buckets = {}
    for ts, o, h, l, c in rows:
        b = (ts // ms) * ms
        if b not in buckets:
            buckets[b] = [o, h, l, c]
        else:
            x = buckets[b]
            x[1] = max(x[1], h); x[2] = min(x[2], l); x[3] = c
    return [tuple(buckets[k]) for k in sorted(buckets)]


def level_sizes(prices, pcts, total_q, mode):
    if mode == "literal_pct":
        quotes = [total_q * p / 100.0 for p in pcts]
    elif mode == "weights":
        s = sum(pcts) or 1.0
        quotes = [total_q * (p / s) for p in pcts]
    else:
        raise ValueError(mode)
    return [q / pr for q, pr in zip(quotes, prices)], quotes


def simulate(candles, cfg, fee, amounts_mode, rearm, path_conv, bar_seconds):
    start_price = candles[0][0]
    total_q     = cfg["total_amount_quote"]
    claimed_q   = cfg.get("claimed_base_value_quote", 0.0)
    # cooldown expressed in BARS, derived from the bar duration (the v3 fix)
    cooldown_bars = cfg.get("cooldown_time", 0.0) / bar_seconds

    bp, bpct = cfg["buy_prices"],  cfg["buy_amounts_pct"]
    sp, spct = cfg["sell_prices"], cfg["sell_amounts_pct"]
    bqty, bq = level_sizes(bp, bpct, total_q, amounts_mode)
    sqty, sq = level_sizes(sp, spct, total_q, amounts_mode)

    base  = claimed_q / start_price
    quote = total_q
    init_value = quote + base * start_price

    b_arm = [True] * len(bp); b_last = [-1e9] * len(bp); b_fill = [0] * len(bp)
    s_arm = [True] * len(sp); s_last = [-1e9] * len(sp); s_fill = [0] * len(sp)
    fees_paid = bought_q = sold_q = 0.0
    skipped_buys = 0
    eq = []

    for t, (o, h, l, c) in enumerate(candles):
        if path_conv == "standard":
            path = [o, l, h, c] if c >= o else [o, h, l, c]
        else:
            path = [o, h, l, c] if c >= o else [o, l, h, c]
        for a, bb in zip(path, path[1:]):
            if bb < a:
                for i, L in enumerate(bp):
                    if b_arm[i] and bb <= L <= a:
                        cost = L * bqty[i]; f = cost * fee
                        if quote >= cost + f:
                            quote -= cost + f; base += bqty[i]
                            fees_paid += f; bought_q += cost
                            b_arm[i] = False; b_last[i] = t; b_fill[i] += 1
                        else:
                            skipped_buys += 1
            elif bb > a:
                for i, L in enumerate(sp):
                    if s_arm[i] and a <= L <= bb and base >= sqty[i]:
                        proceeds = L * sqty[i]; f = proceeds * fee
                        quote += proceeds - f; base -= sqty[i]
                        fees_paid += f; sold_q += proceeds
                        s_arm[i] = False; s_last[i] = t; s_fill[i] += 1
        if rearm:
            for i, L in enumerate(bp):
                if not b_arm[i] and c > L and (t - b_last[i]) >= cooldown_bars:
                    b_arm[i] = True
            for i, L in enumerate(sp):
                if not s_arm[i] and c < L and (t - s_last[i]) >= cooldown_bars:
                    s_arm[i] = True
        eq.append(quote + base * c)

    last = candles[-1][3]
    final_value = quote + base * last
    peak = -1e18; mdd = 0.0
    for e in eq:
        peak = max(peak, e); mdd = max(mdd, (peak - e) / peak if peak > 0 else 0)
    return {
        "init": init_value, "final": final_value,
        "ret_%": (final_value - init_value) / init_value * 100,
        "maxDD_%": mdd * 100,
        "endInv_%": (base * last) / final_value * 100 if final_value else 0,
        "end_quote": quote, "fees": fees_paid,
        "bought": bought_q, "sold": sold_q, "skipped_buys": skipped_buys,
        "cooldown_bars": cooldown_bars,
        "buy_fills": b_fill, "sell_fills": s_fill,
        "buy_quote": bq, "sell_quote": sq,
        "buy_prices": bp, "sell_prices": sp,
    }


def report(cfg, r):
    print("=" * 68)
    print(f"  {cfg['name']}   ({INTERVAL} bars, amounts={AMOUNTS_MODE}, "
          f"rearm={REARM}, fee={FEE_RATE})")
    print("=" * 68)
    print(f"  initial ${r['init']:.2f} -> final ${r['final']:.2f}    RETURN {r['ret_%']:+.2f}%")
    print(f"  maxDD {r['maxDD_%']:.1f}%   endInv {r['endInv_%']:.1f}%   "
          f"end quote ${r['end_quote']:.0f}   fees ${r['fees']:.2f}")
    print(f"  gross bought ${r['bought']:.0f}   gross sold ${r['sold']:.0f}   "
          f"skipped (no cash): {r['skipped_buys']}   cooldown={r['cooldown_bars']:.0f} bars")
    print("  -- BUY rungs (price | $size | fills) --")
    for p, q, f in zip(r["buy_prices"], r["buy_quote"], r["buy_fills"]):
        print(f"     {p:>7.2f} | ${q:>6.2f} | {f:>4} fills{'   <-- DEAD' if f == 0 else ''}")
    print("  -- SELL rungs (price | $size | fills) --")
    for p, q, f in zip(r["sell_prices"], r["sell_quote"], r["sell_fills"]):
        print(f"     {p:>7.2f} | ${q:>6.2f} | {f:>4} fills{'   <-- DEAD' if f == 0 else ''}")
    print()


if __name__ == "__main__":
    print(f"Fetching MEXC {COIN} {INTERVAL} ...")
    raw = fetch_mexc(COIN, DAYS, INTERVAL)
    candles = resample(raw, INTERVAL_SECONDS)
    print(f"Raw {len(raw)} -> {len(candles)} {INTERVAL} candles "
          f"(expected ~{DAYS*86400//INTERVAL_SECONDS})")
    print(f"first open ${candles[0][0]:.2f} | last close ${candles[-1][3]:.2f} | "
          f"high ${max(c[1] for c in candles):.2f} | low ${min(c[2] for c in candles):.2f}\n")

    runs = []
    for cfg in (CFG_V1, CFG_V2):
        r = simulate(candles, cfg, FEE_RATE, AMOUNTS_MODE, REARM, PATH_CONV, INTERVAL_SECONDS)
        report(cfg, r); runs.append((cfg["name"], r))

    print("SUMMARY")
    print(f"  {'config':<24}{'ret%':>8}{'maxDD%':>9}{'endInv%':>9}{'skipBuy':>9}")
    for name, r in runs:
        print(f"  {name:<24}{r['ret_%']:>8.2f}{r['maxDD_%']:>9.1f}"
              f"{r['endInv_%']:>9.1f}{r['skipped_buys']:>9}")

    # backtest a new config:
    # my = parse_config(open("range_inventory_ladder_xmr_V3.yml").read(), "V3")
    # report(my, simulate(candles, my, FEE_RATE, AMOUNTS_MODE, REARM, PATH_CONV, INTERVAL_SECONDS))

Fetching MEXC XMRUSDT 5m ...
Raw 100000 -> 100000 5m candles (expected ~103680)
first open $312.81 | last close $387.85 | high $800.36 | low $229.91

  V1 (current)   (5m bars, amounts=literal_pct, rearm=True, fee=0.002)
  initial $900.00 -> final $1415.38    RETURN +57.26%
  maxDD 33.1%   endInv 82.7%   end quote $244   fees $19.76
  gross bought $5109   gross sold $4773   skipped (no cash): 2192   cooldown=12 bars
  -- BUY rungs (price | $size | fills) --
      328.00 | $  3.00 |  212 fills
      324.00 | $  6.00 |  139 fills
      321.00 | $  6.00 |   85 fills
      318.00 | $ 24.00 |   67 fills
      315.00 | $ 21.00 |   52 fills
      312.00 | $  6.00 |   49 fills
      305.00 | $  3.00 |   45 fills
  -- SELL rungs (price | $size | fills) --
      331.50 | $  3.00 |  183 fills
      335.00 | $  6.00 |  162 fills
      340.00 | $  6.00 |  141 fills
      345.00 | $ 12.00 |  107 fills
      350.00 | $ 12.00 |   66 fills
      355.00 | $  6.00 |   35 fills
      360.00 | $  6.00 |   